In [ ]:
# pip install --upgrade ipywidgets
# pip install realtabformer
# pip install datasets==2.6.1

In [1]:
import os
import pandas as pd
from pathlib import Path
from datetime import datetime
import sys
sys.path.insert(0, '/home/yjung/SynPriv')
from src.realtabformer import REaLTabFormer
import warnings
warnings.filterwarnings("ignore")
device = 'cuda:3'

exp_name = "rtf_retail"
sample_sizes = [5000, 10000]
data_path = Path("../data/sampled")
base_save_dir = Path(f"../results/{exp_name}")
join_on = "fkey"
n_samples = 1000  # 생성할 메타 테이블 샘플 수

def run_rtf(sample_size):
    sub_exp_name = f"train{sample_size}"
    save_dir = base_save_dir / sub_exp_name
    save_dir.mkdir(parents=True, exist_ok=True)

    meta_path = data_path / f"retail_sample_{sample_size}_meta.csv"
    flow_path = data_path / f"retail_sample_{sample_size}_flow.csv"
    parent_df = pd.read_csv(meta_path)
    child_df = pd.read_csv(flow_path)

    print(f"{sample_size} parent 학습")
    parent_model = REaLTabFormer(model_type="tabular", epochs=10)
    parent_model.fit(parent_df.drop(columns=[join_on]), device=device)
    parent_model_dir = save_dir / "rtf_parent"
    parent_model.save(parent_model_dir)

    print(f"{sample_size} child 학습")
    parent_model_path = sorted(parent_model_dir.glob("id*"), key=os.path.getmtime)[-1]
    child_model = REaLTabFormer(model_type="relational", parent_realtabformer_path=parent_model_path, output_max_length=1024, epochs=10)
    child_model.fit(df=child_df, in_df=parent_df, join_on=join_on, device=device)
    child_model_dir = save_dir / "rtf_child"
    child_model.save(child_model_dir)

    print(f"{sample_size} 샘플 생성")
    parent_samples = parent_model.sample(n_samples)
    parent_samples.index.name = join_on
    parent_samples = parent_samples.reset_index()

    child_samples = child_model.sample(
        input_unique_ids=parent_samples[join_on],
        input_df=parent_samples.drop(columns=[join_on]),
        gen_batch=64
    )

    print(f"{sample_size} 샘플 저장")
    parent_samples.to_csv(save_dir / f"syn_meta_{n_samples}.csv", index=False)
    child_samples_ = child_samples.reset_index().rename(columns={"index": join_on})
    child_samples_.to_csv(save_dir / f"syn_flow_{n_samples}.csv", index=False)

    syn_df = parent_samples.merge(child_samples_, on=join_on, how="inner")
    syn_df.to_csv(save_dir / f"syn_rtf_{n_samples}.csv", index=False)
    print(f"{sample_size} 완료: {save_dir}")

for size in sample_sizes:
    run_rtf(size)

5000 parent 학습
Computing the sensitivity threshold...
Using parallel computation!!!


Bootstrap round:   0%|          | 0/500 [00:00<?, ?it/s]

Sensitivity threshold summary:
count    500.000000
mean       0.000356
std        0.002593
min       -0.006570
25%       -0.001470
50%        0.000319
75%        0.002109
max        0.009229
dtype: float64
Sensitivity threshold: 0.004601478196600145 qt_max: 0.05


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


...

  0%|          | 0/1000 [00:00<?, ?it/s]

Generated 0 invalid samples out of total 1024 samples generated. Sampling efficiency is: 100.0000%


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

  0%|          | 0/16 [00:00<?, ?it/s]

10000 샘플 저장
10000 완료: ../results/rtf_retail/train10000
